In [2]:
#import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score)
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier


In [3]:
df = pd.read_csv(r'D:\projects\commerce dataset\E-commerce\data\processed\modeling_dataset.csv')

In [4]:
x = df.drop(columns=['return_fraud'])
y = df['return_fraud']

x_train, x_test, y_train, y_test = train_test_split(
    x,y,
    test_size=0.2, random_state=42, stratify=y
)
print("Train:", x_train.shape, "Test:", x_test.shape)

Train: (40000, 41) Test: (10000, 41)


In [5]:

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)   
x_test_scaled  = scaler.transform(x_test)        

## Model Building

1.Logistic Regression

In [6]:
#Train
logreg = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
logreg.fit(x_train_scaled, y_train)

#Test
y_pred = logreg.predict(x_test_scaled)     # 0 or 1
y_proba = logreg.predict_proba(x_test_scaled)[:, 1]   # fraud probability (0 to 1)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_proba))

Accuracy : 0.9137
Precision: 0.9700991609458429
Recall   : 0.9050668943922573
F1 Score : 0.9364553420219425
ROC-AUC  : 0.9754186151887377


2.Random Forest

In [10]:


# Random Forest - NO scaling (tree-based)
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced',
                            random_state=42, n_jobs=-1)
rf.fit(x_train, y_train)   # raw X_train (NOT scaled)

y_pred  = rf.predict(x_test)
y_proba = rf.predict_proba(x_test)[:, 1]

# Overfit check: train vs test + cross-validation
print("Train F1:", f1_score(y_train, rf.predict(x_train)))
print("Test F1 :", f1_score(y_test, y_pred))
cv = cross_val_score(rf, x, y, cv=5, scoring='f1')
print("CV F1   :", cv.mean(), "+/-", cv.std())

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_proba))

Train F1: 1.0
Test F1 : 0.9996443053283062
CV F1   : 0.9997296477487033 +/- 0.00015840156777031193
Accuracy : 0.9995
Precision: 0.9992888636040392
Recall   : 1.0
F1 Score : 0.9996443053283062
ROC-AUC  : 0.9999025858608367


3.XG Boost

In [ ]:
# scale_pos_weight = genuine count / fraud count (imbalance handle)
spw = (y_train == 0).sum() / (y_train == 1).sum()

xgb = XGBClassifier(n_estimators=100, scale_pos_weight=spw,
                    random_state=42, eval_metric='logloss', n_jobs=-1)
xgb.fit(X_train, y_train) #raw X_train (NOT scaled)

y_pred = xgb.predict(X_test)
y_proba = xgb.predict_proba(X_test)[:, 1]

print("=== XGBOOST — Test Results ===")
print(f"Accuracy : {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall   : {recall_score(y_test, y_pred):.3f}")
print(f"F1 Score : {f1_score(y_test, y_pred):.3f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_proba):.3f}")

=== XGBOOST — Test Results ===
Accuracy : 1.000
Precision: 0.999
Recall   : 1.000
F1 Score : 1.000
ROC-AUC  : 1.000
